# Notebook 03: Tool Design Pattern

The CCA exam tests whether you understand that **tool selection accuracy degrades measurably when an agent has more than 4–5 tools**. This notebook shows both patterns on the same scenario — a \$600 refund for customer C003 — to demonstrate that 15 overlapping tools cause tool selection errors that 5 focused tools avoid.

Pattern covered: **5-tool focused agent with negative bounds** vs. 15-tool Swiss Army agent with overlapping descriptions.

## Setup

In [ ]:
import sys
from pathlib import Path

# Add project root so helpers and customer_service are importable
sys.path.insert(0, str(Path(".").resolve()))

import anthropic
from helpers import compare_results, print_usage

from customer_service.anti_patterns.swiss_army_agent import SWISS_ARMY_TOOLS
from customer_service.data.customers import CUSTOMERS
from customer_service.data.scenarios import SCENARIOS
from customer_service.services.audit_log import AuditLog
from customer_service.services.container import ServiceContainer
from customer_service.services.customer_db import CustomerDatabase
from customer_service.services.escalation_queue import EscalationQueue
from customer_service.services.financial_system import FinancialSystem
from customer_service.services.policy_engine import PolicyEngine
from customer_service.tools.definitions import TOOLS

In [ ]:
def make_services() -> ServiceContainer:
    """Create a fresh ServiceContainer with seed customer data."""
    return ServiceContainer(
        customer_db=CustomerDatabase(CUSTOMERS),
        policy_engine=PolicyEngine(),
        financial_system=FinancialSystem(),
        escalation_queue=EscalationQueue(),
        audit_log=AuditLog(),
    )


client = anthropic.Anthropic()
scenario = SCENARIOS["amount_threshold"]  # C003, $600 — same as Notebook 01
print(f"Scenario: {scenario['customer_id']} - {scenario['message']}")
print("\nThis is the SAME scenario as Notebook 01, demonstrating a different CCA failure mode.")

In [ ]:
# Show the tool count difference before running any agent
correct_tool_names = {t["name"] for t in TOOLS}
print(f"Correct agent: {len(TOOLS)} tools")
print(f"Swiss Army agent: {len(SWISS_ARMY_TOOLS)} tools")
print("\nCorrect tool names:")
for t in TOOLS:
    print(f"  - {t['name']}")
print("\nSwiss Army tool names (10 distractors added):")
for t in SWISS_ARMY_TOOLS:
    marker = "" if t["name"] in correct_tool_names else " <-- distractor"
    print(f"  - {t['name']}{marker}")

## Anti-Pattern: Swiss Army Agent (15 Tools)

<div style="border-left: 4px solid #dc3545; padding: 12px 16px; background: #fff5f5; margin: 8px 0;">
<strong>What's wrong:</strong> The Swiss Army agent has 15 tools with overlapping descriptions. Tools like <code>file_billing_dispute</code> sound similar to <code>process_refund</code> for a dollar-amount issue, and <code>create_support_ticket</code> overlaps with <code>escalate_to_human</code> for closure or legal cases. Claude mis-routes to distractor tools instead of the correct ones, causing observable failures.
</div>

In [ ]:
from customer_service.anti_patterns.swiss_army_agent import run_swiss_army_agent

anti_services = make_services()
print("Running anti-pattern (Swiss Army agent, 15 tools)...")
anti_result = run_swiss_army_agent(client, anti_services, scenario["message"])
print(f"Stop reason: {anti_result.stop_reason}")
print(f"Tool calls: {[tc['name'] for tc in anti_result.tool_calls]}")

# Check if any distractor tools were called
anti_distractor_calls = [
    tc["name"] for tc in anti_result.tool_calls if tc["name"] not in correct_tool_names
]
if anti_distractor_calls:
    print(f"\nDISTRACTOR TOOLS CALLED: {anti_distractor_calls}")
    print("Tool selection error: agent chose wrong tool from 15-tool set")
else:
    print("\nNote: Agent selected correct tools this run (probabilistic)")

In [ ]:
class _UsageWrapper:
    def __init__(self, u):
        self.usage = u


print_usage(_UsageWrapper(anti_result.usage))

## Correct Pattern: Focused 5-Tool Agent

<div style="border-left: 4px solid #28a745; padding: 12px 16px; background: #f0fff4; margin: 8px 0;">
<strong>Why this works:</strong> The focused agent has exactly 5 tools, each with precise descriptions and negative bounds (e.g., "does NOT handle billing disputes"). With no overlapping tools, Claude selects correctly every time. If the task genuinely requires more capabilities, the correct CCA pattern is coordinator-subagent — not cramming more tools into one agent.
</div>

In [ ]:
from customer_service.agent import get_system_prompt, run_agent_loop

correct_services = make_services()
print("Running correct pattern (focused 5-tool agent)...")
correct_result = run_agent_loop(
    client,
    correct_services,
    scenario["message"],
    get_system_prompt(),
)
print(f"Stop reason: {correct_result.stop_reason}")
print(f"Tool calls: {[tc['name'] for tc in correct_result.tool_calls]}")

# All calls should be from the correct tool set
correct_distractor_calls = [
    tc["name"] for tc in correct_result.tool_calls if tc["name"] not in correct_tool_names
]
if not correct_distractor_calls:
    print("\nSUCCESS: All tool calls from correct 5-tool set")
else:
    print(f"\nUNEXPECTED distractor calls: {correct_distractor_calls}")

In [ ]:
print_usage(_UsageWrapper(correct_result.usage))

## Compare Results

In [ ]:
compare_results(
    {
        "tool_count_available": len(SWISS_ARMY_TOOLS),
        "distractor_tools_called": len(anti_distractor_calls),
        "tool_calls_total": len(anti_result.tool_calls),
    },
    {
        "tool_count_available": len(TOOLS),
        "distractor_tools_called": len(correct_distractor_calls),
        "tool_calls_total": len(correct_result.tool_calls),
    },
)

> **CCA Exam Tip:** Beyond 4–5 tools per agent, tool selection accuracy degrades measurably. If an exam scenario describes an agent with 12+ tools and selection errors, the answer is to split into focused agents with 4–5 tools each using the **coordinator-subagent pattern** — NOT to improve tool descriptions or add a disambiguation prompt. Adding tools to a single agent is always the wrong answer.

## Summary

- **Anti-pattern failure:** A 15-tool Swiss Army agent with overlapping descriptions causes tool selection errors. Claude routes to distractor tools like `file_billing_dispute` instead of `process_refund`, or to `create_support_ticket` instead of `escalate_to_human`.
- **Correct pattern guarantee:** A focused 5-tool agent with precise descriptions and negative bounds gives Claude unambiguous choices. Tool selection is accurate because there is no ambiguity.
- **Key principle:** When a task requires more than 5 tools, the CCA correct answer is the coordinator-subagent pattern — a coordinator agent that routes to specialized sub-agents, each with 4–5 focused tools.